### Initialization

In [ ]:
import pandas as pd
from datasets import load_dataset
from torchao.quantization import (
    Int8DynamicActivationInt8WeightConfig,
    Int8WeightOnlyConfig,
)
import torch

from sequence_tagging_benchmark.taggers import (
    NltkHMMTagger,
    HmmlearnTagger,
    CRFTagger,
    BiLSTMTagger,
    TransformerTagger,
    QuantizedBiLSTMTagger,
    QuantizedTransformerTagger,
    ONNXTransformerTagger,
)

In [ ]:
NUM_THREADS = 8

torch.set_num_threads(NUM_THREADS)

### Data loading

In [ ]:
dataset = load_dataset("universal_dependencies", "en_ewt")

train_data = dataset["train"]
test_data = dataset["test"]

upos_feature = train_data.features["upos"].feature
int2str = upos_feature.int2str

In [ ]:
raw_train_tokens = [row["tokens"] for row in train_data]
raw_train_tags = [[int2str(tag_id) for tag_id in row["upos"]] for row in train_data]

raw_test_tokens = [row["tokens"] for row in test_data]
raw_test_tags = [[int2str(tag_id) for tag_id in row["upos"]] for row in test_data]

### Benchmarking

In [ ]:
def clean_transformer_upos_tag(raw_tag: str) -> str:
    if raw_tag.startswith("B-") or raw_tag.startswith("I-"):
        raw_tag = raw_tag[2:]
    base_tag = raw_tag.split("|")[0]
    base_tag = base_tag.split("+")[0]
    return base_tag

In [ ]:
TAG_TYPE = "upos"
BLSTM_MODEL_NAME = "flair/upos-multi-fast"
TRANSFORMER_MODEL_NAME = "KoichiYasuoka/roberta-base-english-upos"

nltk_hmm = NltkHMMTagger()
nltk_hmm.train(raw_train_tokens, raw_train_tags)

hmmlearn_hmm = HmmlearnTagger()
hmmlearn_hmm.train(raw_train_tokens, raw_train_tags)

crf = CRFTagger()
crf.train(raw_train_tokens, raw_train_tags)

bilstm = BiLSTMTagger(model_name=BLSTM_MODEL_NAME, tag_type=TAG_TYPE)
bilstm.train(raw_train_tokens, raw_train_tags)

dynamic_quantized_bilstm = QuantizedBiLSTMTagger(
    model_name=BLSTM_MODEL_NAME,
    tag_type=TAG_TYPE,
    quantize_config=Int8DynamicActivationInt8WeightConfig(),
)
dynamic_quantized_bilstm.train(raw_train_tokens, raw_train_tags)

weight_only_quantized_bilstm = QuantizedBiLSTMTagger(
    model_name=BLSTM_MODEL_NAME,
    tag_type=TAG_TYPE,
    quantize_config=Int8WeightOnlyConfig(),
)
weight_only_quantized_bilstm.train(raw_train_tokens, raw_train_tags)

transformer = TransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    tag_clean_func=clean_transformer_upos_tag,
)
transformer.train(raw_train_tokens, raw_train_tags)

dynamic_quantized_transformer = QuantizedTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    tag_clean_func=clean_transformer_upos_tag,
    quantize_config=Int8DynamicActivationInt8WeightConfig(),
)
dynamic_quantized_transformer.train(raw_train_tokens, raw_train_tags)

weight_only_quantized_transformer = QuantizedTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    tag_clean_func=clean_transformer_upos_tag,
    quantize_config=Int8WeightOnlyConfig(),
)
weight_only_quantized_transformer.train(raw_train_tokens, raw_train_tags)

onnx_transformer = ONNXTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    tag_clean_func=clean_transformer_upos_tag,
)
onnx_transformer.train(raw_train_tokens, raw_train_tags)

In [ ]:
# Evaluate all models
results = []

# Traditional models
results.append(
    {
        "name": "nltk_hmm",
        **nltk_hmm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "hmmlearn_hmm",
        **hmmlearn_hmm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {"name": "crf", **crf.evaluate(raw_test_tokens, raw_test_tags, batch_size=1)}
)

# BiLSTM variants
results.append(
    {
        "name": "bilstm-batch-1",
        **bilstm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "bilstm-batch-128",
        **bilstm.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results.append(
    {
        "name": "bilstm-batch-128-smart-batching",
        **bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128, use_smart_batching=True
        ),
    }
)
results.append(
    {
        "name": "dynamic-quantized-bilstm-batch-128",
        **dynamic_quantized_bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "weight-only-quantized-bilstm-batch-128",
        **weight_only_quantized_bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)

# Transformer variants
results.append(
    {
        "name": "transformer-batch-1",
        **transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "transformer-batch-128",
        **transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results.append(
    {
        "name": "transformer-batch-128-smart-batching",
        **transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128, use_smart_batching=True
        ),
    }
)
results.append(
    {
        "name": "dynamic-quantized-transformer-batch-128",
        **dynamic_quantized_transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "weight-only-quantized-transformer-batch-128",
        **weight_only_quantized_transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "onnx-transformer-batch-128",
        **onnx_transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results = pd.DataFrame(results)

In [ ]:
results.to_csv("../artifacts/pos_tagging_results.csv")